In [ ]:
import os
import sys
import copy
from collections.abc import Callable
from enum import Enum
from pathlib import Path
import concurrent.futures
import time

from dotenv import load_dotenv
from google.genai import Client, types
from pydantic import BaseModel, Field

from test_validate_plan import validate_plan

load_dotenv()

True

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-2.5-flash"

### Task 1

In [5]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"
    SKIPPED = "skipped"


class PlanStep(BaseModel):
    id: str = Field(description="unique id for each plansstep of plan")
    description: str = Field(description="Description for planstep")
    tool_name_hint: str | None = Field(default=None, description="Tool hint for this step")
    depends_on: list[str] | None = Field(
        default_factory=list, description="Id's of previous depended steps"
    )
    success_criteria: str = Field(description="criterias for mark success")
    status: StepStatus = Field(default=StepStatus.PENDING, description="Status of this step")
    result_summary: str | None = Field(default=None, description="Summary of this step")
    attempts: int = Field(default=0, description="Attempts to complete this step")


class Plan(BaseModel):
    goal: str = Field(description="Goal for the agent")
    steps: list[PlanStep] = Field(description="Steps to complete goal")
    version: int = Field(default=1, description="Version of this plan")
    assumptions: list[str] = Field(default_factory=list, description="Assumptions for this goals")

In [15]:
goal = "Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS"

response = google_client.models.generate_content(
    model=MODEL,
    contents=f"Generate a detailed execution plan for the following goal: {goal}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Plan,
        system_instruction=(
            "You are an expert AI planning agent. Given a user goal, break it down into "
            "logical, atomic, executable PlanSteps with explicit dependencies, tool hints, "
            "and clear success criteria."
        ),
    ),
)

plan: Plan = response.parsed

print(f"Goal: {plan.goal}")
print(f"Version: {plan.version}")
print(f"Assumptions: {plan.assumptions}\n")
print("Execution Steps:")
for step in plan.steps:
    deps = f" (Depends on: {', '.join(step.depends_on)})" if step.depends_on else ""
    tool = f" [Tool: {step.tool_name_hint}]" if step.tool_name_hint else ""
    print(f"  [{step.id}] {step.description}{tool}{deps}")
    print(f"       Success Criteria: {step.success_criteria}")

Goal: Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS
Version: 1
Assumptions: ['AWS account with appropriate permissions is available', 'GitHub repository is configured for storing source code and running Actions', 'Terraform state management backend is accessible']

Execution Steps:
  [step_1] Develop the FastAPI microservice application code with database connection logic and endpoints [Tool: IDE / File System]
       Success Criteria: FastAPI application is fully written with main.py, models.py, requirements.txt, and passing unit tests
  [step_2] Create a multi-stage Dockerfile for containerizing the FastAPI application [Tool: Docker] (Depends on: step_1)
       Success Criteria: Dockerfile builds successfully and the container runs locally with the FastAPI app responding
  [step_3] Define Docker Compose setup for local testing of FastAPI and PostgreSQL database [Tool: Docker Compose] (Depends on: step_2)
       Success Crite

### Task 2

implemented in test_validate_plan.py

### Task 3

In [ ]:
# Ensure local directory is accessible for imports
nb_dir = str(Path.cwd())
if nb_dir not in sys.path:
    sys.path.append(nb_dir)


def get_ready_steps(plan: Plan) -> list[PlanStep]:
    """Returns all steps whose status is PENDING and whose dependencies are all COMPLETED."""
    completed_ids = {s.id for s in plan.steps if s.status == StepStatus.COMPLETED}
    return [
        step
        for step in plan.steps
        if step.status == StepStatus.PENDING
        and all(dep in completed_ids for dep in (step.depends_on or []))
    ]


def default_executor(step: PlanStep) -> str:
    """Mock execution for a plan step simulating successful action."""
    tool_info = f" via {step.tool_name_hint}" if step.tool_name_hint else ""
    return f"Completed '{step.id}'{tool_info}. Criteria satisfied: {step.success_criteria}"


def dag_scheduler(
    plan: Plan,
    executor: Callable[[PlanStep], str] | None = None,
    stop_on_failure: bool = True,
    verbose: bool = True,
) -> Plan:
    """
    Executes ready steps in dependency order sequentially.

    At each iteration:
      1. Cascades SKIPPED status to any pending step whose dependency failed or was skipped.
      2. Finds all ready steps (PENDING with all dependencies COMPLETED).
      3. Runs ready steps sequentially.
      4. Repeats until all steps reach a terminal status.
    """
    validate_plan(plan)
    exec_fn = executor or default_executor

    if verbose:
        print(f"=== Starting Sequential DAG Scheduler for: '{plan.goal}' ===\n")

    iteration = 1
    while True:
        # 1. Cascade SKIPPED status if any dependency failed or was skipped
        blocked_ids = {
            s.id for s in plan.steps if s.status in (StepStatus.FAILED, StepStatus.SKIPPED)
        }
        for step in plan.steps:
            if step.status == StepStatus.PENDING:
                deps = step.depends_on or []
                if any(dep in blocked_ids for dep in deps):
                    step.status = StepStatus.SKIPPED
                    step.result_summary = "Skipped due to blocked dependency."
                    if verbose:
                        print(f"  ⏭ [{step.id}] SKIPPED: prerequisite failed or skipped")

        # 2. Identify ready steps
        ready_steps = get_ready_steps(plan)

        if not ready_steps:
            # Check if all steps reached terminal status
            pending_steps = [s.id for s in plan.steps if s.status == StepStatus.PENDING]
            if pending_steps:
                raise RuntimeError(
                    f"Deadlock! No ready steps, but steps remain pending: {pending_steps}"
                )
            break

        if verbose:
            print(f"[Iteration {iteration}] Ready steps: {[s.id for s in ready_steps]}")

        # 3. Execute ready steps sequentially
        for step in ready_steps:
            deps_str = (
                f" (Depends on: {', '.join(step.depends_on)})"
                if step.depends_on
                else " (Root step)"
            )
            if verbose:
                print(f"  ▶ Running [{step.id}]: {step.description[:55]}...{deps_str}")

            step.status = StepStatus.RUNNING
            step.attempts += 1
            try:
                result = exec_fn(step)
                step.status = StepStatus.COMPLETED
                step.result_summary = result
                if verbose:
                    print(f"    ✓ COMPLETED: {result}")
            except Exception as e:  # noqa: BLE001
                step.status = StepStatus.FAILED
                step.result_summary = f"Error: {e}"
                if verbose:
                    print(f"    ✗ FAILED: {e}")
                if stop_on_failure:
                    if verbose:
                        print("  ⛔ Stopping execution due to step failure.\n")
                    return plan

        iteration += 1
        if verbose:
            print()

    if verbose:
        completed = sum(1 for s in plan.steps if s.status == StepStatus.COMPLETED)
        print(
            f"\n=== Execution Finished: {completed}/{len(plan.steps)} steps completed successfully ==="
        )

    return plan

In [17]:
# Reset statuses for our LLM-generated plan from Task 1 (in case re-running)
for step in plan.steps:
    step.status = StepStatus.PENDING
    step.attempts = 0
    step.result_summary = None

# Execute the plan using our sequential DAG scheduler
executed_plan = dag_scheduler(plan)

# Display execution summary
print(f"\n{'=' * 75}")
print(f"{'Step ID':<10} | {'Status':<12} | {'Attempts':<8} | {'Result Summary'}")
print(f"{'-' * 75}")
for step in executed_plan.steps:
    print(f"{step.id:<10} | {step.status.value:<12} | {step.attempts:<8} | {step.result_summary}")
print(f"{'=' * 75}")

=== Starting Sequential DAG Scheduler for: 'Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS' ===

[Iteration 1] Ready steps: ['step_1', 'step_4']
  ▶ Running [step_1]: Develop the FastAPI microservice application code with ... (Root step)
    ✓ COMPLETED: Completed 'step_1' via IDE / File System. Criteria satisfied: FastAPI application is fully written with main.py, models.py, requirements.txt, and passing unit tests
  ▶ Running [step_4]: Provision AWS infrastructure (RDS PostgreSQL instance, ... (Root step)
    ✓ COMPLETED: Completed 'step_4' via Terraform / AWS CLI. Criteria satisfied: Terraform apply completes without error, creating the ECR repository, RDS database, and ECS service infrastructure

[Iteration 2] Ready steps: ['step_2']
  ▶ Running [step_2]: Create a multi-stage Dockerfile for containerizing the ... (Depends on: step_1)
    ✓ COMPLETED: Completed 'step_2' via Docker. Criteria satisfied: Dockerfile builds succe

### Task 4

In [ ]:
def concurrent_dag_scheduler(
    plan: Plan,
    executor: Callable[[PlanStep], str] | None = None,
    max_concurrency: int = 3,
    stop_on_failure: bool = True,
    verbose: bool = True,
) -> Plan:
    """
    Executes independent ready steps concurrently with a strict concurrency cap.

    As steps complete, newly unblocked ready steps are immediately submitted
    to available worker slots without waiting for entire batches to finish.

    Args:
        plan: The Plan to execute.
        executor: Callable taking a PlanStep and returning a result string.
        max_concurrency: Maximum number of steps executing simultaneously.
        stop_on_failure: Whether to halt on step failure.
        verbose: Whether to print live execution logs.

    Returns:
        The updated Plan with execution results and statuses.
    """
    validate_plan(plan)
    exec_fn = executor or default_executor

    if verbose:
        print(
            f"=== Starting Concurrent DAG Scheduler (Max Concurrency = {max_concurrency}) for: '{plan.goal}' ===\n"
        )

    completed_ids = {s.id for s in plan.steps if s.status == StepStatus.COMPLETED}

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_concurrency) as pool:
        active_futures: dict[concurrent.futures.Future, PlanStep] = {}

        while True:
            # 1. Cascade SKIPPED status if any dependency failed or was skipped
            blocked_ids = {
                s.id for s in plan.steps if s.status in (StepStatus.FAILED, StepStatus.SKIPPED)
            }
            for step in plan.steps:
                if step.status == StepStatus.PENDING:
                    deps = step.depends_on or []
                    if any(dep in blocked_ids for dep in deps):
                        step.status = StepStatus.SKIPPED
                        step.result_summary = "Skipped due to blocked dependency."
                        if verbose:
                            print(f"  ⏭ [{step.id}] SKIPPED: prerequisite failed or skipped")

            # 2. Find newly ready steps that are PENDING and not currently running
            running_ids = {s.id for s in active_futures.values()}
            ready_steps = [
                step
                for step in plan.steps
                if step.status == StepStatus.PENDING
                and step.id not in running_ids
                and all(dep in completed_ids for dep in (step.depends_on or []))
            ]

            # 3. Dispatch ready steps up to the concurrency cap
            for step in ready_steps:
                if len(active_futures) >= max_concurrency:
                    break  # Concurrency cap reached; wait for a slot to free up

                step.status = StepStatus.RUNNING
                step.attempts += 1
                if verbose:
                    deps_str = (
                        f" (after: {', '.join(step.depends_on)})"
                        if step.depends_on
                        else " (Root step)"
                    )
                    slot_info = f"[Slot {len(active_futures) + 1}/{max_concurrency}]"
                    print(
                        f"  ▶ {slot_info} Dispatched [{step.id}]: {step.description[:50]}...{deps_str}"
                    )

                future = pool.submit(exec_fn, step)
                active_futures[future] = step

            # 4. If no tasks are running and no ready steps: check completion or deadlock
            if not active_futures:
                pending_steps = [s.id for s in plan.steps if s.status == StepStatus.PENDING]
                if pending_steps:
                    raise RuntimeError(
                        f"Deadlock detected! No active tasks or ready steps, but steps remain pending: {pending_steps}"
                    )
                break

            # 5. Wait for at least ONE running future to complete before re-evaluating DAG
            done, _ = concurrent.futures.wait(
                active_futures.keys(),
                return_when=concurrent.futures.FIRST_COMPLETED,
            )

            for future in done:
                step = active_futures.pop(future)
                try:
                    result = future.result()
                    step.status = StepStatus.COMPLETED
                    step.result_summary = result
                    completed_ids.add(step.id)
                    if verbose:
                        print(f"    ✓ COMPLETED [{step.id}]: {result}")
                except Exception as e:  # noqa: BLE001
                    step.status = StepStatus.FAILED
                    step.result_summary = f"Error: {e}"
                    if verbose:
                        print(f"    ✗ FAILED [{step.id}]: {e}")
                    if stop_on_failure:
                        if verbose:
                            print("  ⛔ Stopping execution due to step failure.\n")
                        for f in active_futures:
                            f.cancel()
                        return plan

    if verbose:
        completed = sum(1 for s in plan.steps if s.status == StepStatus.COMPLETED)
        print(
            f"\n=== Concurrent Execution Finished: {completed}/{len(plan.steps)} steps completed successfully ==="
        )

    return plan

In [ ]:
# Helper to reset plan step states
def reset_plan(p: Plan) -> Plan:
    for s in p.steps:
        s.status = StepStatus.PENDING
        s.attempts = 0
        s.result_summary = None
    return p


# Timed executor simulating realistic I/O work per step
def timed_executor(step: PlanStep, delay: float = 0.4) -> str:
    time.sleep(delay)
    tool = f" via {step.tool_name_hint}" if step.tool_name_hint else ""
    return f"Done [{step.id}]{tool} ({delay}s)"


# Run concurrent scheduler with concurrency cap = 2
reset_plan(plan)
executed_concurrent = concurrent_dag_scheduler(
    plan,
    executor=lambda s: timed_executor(s, delay=0.4),
    max_concurrency=2,
    verbose=True,
)

In [ ]:
def benchmark_speedup(plan_to_test: Plan, step_delay: float = 0.3, concurrency_caps=(2, 3, 5)):
    """Measures and compares execution times and speedups between Sequential and Concurrent execution."""
    print("=" * 75)
    print("BENCHMARK: Sequential vs. Concurrent DAG Execution")
    print(f"Plan: '{plan_to_test.goal}' ({len(plan_to_test.steps)} steps)")
    print(f"Simulated work per step: {step_delay:.2f}s")
    print("=" * 75)

    # 1. Sequential execution benchmark
    seq_plan = copy.deepcopy(plan_to_test)
    reset_plan(seq_plan)

    t0 = time.perf_counter()
    dag_scheduler(
        seq_plan,
        executor=lambda s: timed_executor(s, delay=step_delay),
        verbose=False,
    )
    seq_duration = time.perf_counter() - t0
    print(
        f"\n1. Sequential Execution Time: {seq_duration:.3f}s (Sum of steps = {len(seq_plan.steps) * step_delay:.2f}s)"
    )

    # 2. Concurrent executions across different concurrency caps
    results = []
    for cap in concurrency_caps:
        conc_plan = copy.deepcopy(plan_to_test)
        reset_plan(conc_plan)

        t0 = time.perf_counter()
        concurrent_dag_scheduler(
            conc_plan,
            executor=lambda s: timed_executor(s, delay=step_delay),
            max_concurrency=cap,
            verbose=False,
        )
        conc_duration = time.perf_counter() - t0
        speedup = seq_duration / conc_duration
        pct_reduction = ((seq_duration - conc_duration) / seq_duration) * 100

        results.append(
            {
                "cap": cap,
                "duration": conc_duration,
                "speedup": speedup,
                "reduction": pct_reduction,
            }
        )

    # 3. Formatted comparison table
    print("\n" + "=" * 75)
    print(
        f"{'Execution Mode':<18} | {'Concurrency Cap':<16} | {'Time (s)':<10} | {'Speedup':<10} | {'Time Saved'}"
    )
    print("-" * 75)
    print(
        f"{'Sequential':<18} | {'1 (sequential)':<16} | {seq_duration:<10.3f} | {'1.00x':<10} | {'Baseline'}"
    )

    for r in results:
        print(
            f"{'Concurrent':<18} | {r['cap']:<16} | {r['duration']:<10.3f} | {r['speedup']:<9.2f}x | {r['reduction']:.1f}%"
        )
    print("=" * 75)


# Execute benchmark with simulated latency per step
benchmark_speedup(plan, step_delay=0.3, concurrency_caps=[2, 3, 5])

BENCHMARK: Sequential vs. Concurrent DAG Execution
Plan: 'Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS' (6 steps)
Simulated work per step: 0.30s

1. Sequential Execution Time: 1.801s (Sum of steps = 1.80s)

Execution Mode     | Concurrency Cap  | Time (s)   | Speedup    | Time Saved
---------------------------------------------------------------------------
Sequential         | 1 (sequential)   | 1.801      | 1.00x      | Baseline
Concurrent         | 2                | 1.210      | 1.49     x | 32.8%
Concurrent         | 3                | 1.202      | 1.50     x | 33.2%
Concurrent         | 5                | 1.202      | 1.50     x | 33.2%


### Task 5: Per-Step Success Verification

In [ ]:
class StepVerificationResult(BaseModel):
    """Structured result from the checker-model evaluator."""

    is_successful: bool = Field(
        description="Whether the execution output fully satisfies the step success criteria"
    )
    reasoning: str = Field(
        description="Detailed explanation of why the output passes or fails the criteria"
    )
    feedback_for_retry: str | None = Field(
        default=None, description="Actionable suggestion if the step failed"
    )


def verify_step_programmatic(step: PlanStep, output: str) -> tuple[bool, str]:
    """
    Deterministic rule-based check on step output:
      1. Verifies output is not empty/null.
      2. Ensures output length exceeds minimum threshold.
      3. Checks that no error signatures ('Error:', 'Exception:', 'Traceback') are present.
    """
    if not output or not output.strip():
        return False, "Output is empty or whitespace only."

    if len(output.strip()) < 15:
        return False, f"Output is too short ({len(output.strip())} chars) for verified execution."

    error_signatures = [
        "traceback (most recent call last)",
        "error:",
        "exception:",
        "failed to connect",
    ]
    lower_output = output.lower()
    for sig in error_signatures:
        if sig in lower_output:
            return False, f"Output contains error signature: '{sig}'."

    return True, "Programmatic checks passed: non-empty, error-free output."


def verify_step_with_checker_model(
    step: PlanStep,
    output: str,
    client: Client = google_client,
    model: str = MODEL,
) -> StepVerificationResult:
    """Uses the LLM as a judge to evaluate if output meets success criteria."""
    prompt = f"""You are an objective QA Verification Judge for an AI planning agent.
        Your task is to determine if the actual execution output satisfies the required success criteria.

        Step ID: {step.id}
        Description: {step.description}
        Tool Used: {step.tool_name_hint or "General"}
        Required Success Criteria: {step.success_criteria}

        Actual Execution Output:
        {output}

        Decide if the actual output proves the success criteria has been met.
    """
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=StepVerificationResult,
            system_instruction="You are an impartial, strict verification judge evaluating step execution against criteria.",
        ),
    )
    result: StepVerificationResult = response.parsed
    return result


def verify_step_success(
    step: PlanStep,
    output: str,
) -> tuple[bool, str, StepVerificationResult | None]:
    """
    Per-step verification combining programmatic checks and the checker-model.
      - Stage 1: Fast programmatic validation.
      - Stage 2: Checker-model call if programmatic check passes.
    """
    # 1. Programmatic check
    prog_passed, prog_msg = verify_step_programmatic(step, output)
    if not prog_passed:
        return False, f"[Programmatic Failed] {prog_msg}", None

    # 2. Checker-model call
    model_verif = verify_step_with_checker_model(step, output)
    if not model_verif.is_successful:
        return False, f"[Checker-Model Rejected] {model_verif.reasoning}", model_verif

    return True, f"[Verified] Programmatic OK | Checker Model: {model_verif.reasoning}", model_verif

In [40]:
# Test per-step verification on 3 scenarios using step_1 from the plan:
sample_step = plan.steps[0]
print(f"Testing Verification on Step [{sample_step.id}]: {sample_step.description[:60]}...")
print(f"Target Success Criteria: {sample_step.success_criteria}\n")

# Scenario 1: Valid execution output that genuinely fulfills criteria
valid_output = (
    "Developed FastAPI application in main.py with PostgreSQL endpoints. "
    "Created models.py and requirements.txt. Ran pytest tests/unit - all unit tests passed with 100% success."
)
is_ok, msg, _ = verify_step_success(sample_step, valid_output)
print("Scenario 1 (Valid Execution Output):")
print(f"  Result: {'PASSED ✓' if is_ok else 'FAILED ✗'}")
print(f"  Details: {msg}\n")

# Scenario 2: Programmatically invalid output (contains error signal)
error_output = "Error: Connection refused while connecting to database. Traceback (most recent call last): failed to run tests."
is_ok, msg, _ = verify_step_success(sample_step, error_output)
print("Scenario 2 (Programmatic Failure - Error Caught):")
print(f"  Result: {'PASSED ✓' if is_ok else 'FAILED ✗ (Caught Programmatically)'}")
print(f"  Details: {msg}\n")

# Scenario 3: Programmatically valid, but semantically rejected by checker-model
incomplete_output = "Created an empty main.py file and opened a text editor, but have not implemented any endpoints or unit tests yet."
is_ok, msg, _ = verify_step_success(sample_step, incomplete_output)
print("Scenario 3 (Semantic Failure - Checker Model Rejection):")
print(f"  Result: {'PASSED ✓' if is_ok else 'FAILED ✗ (Caught by Checker Model)'}")
print(f"  Details: {msg}")

Testing Verification on Step [step_1]: Develop the FastAPI microservice application code with datab...
Target Success Criteria: FastAPI application is fully written with main.py, models.py, requirements.txt, and passing unit tests

Scenario 1 (Valid Execution Output):
  Result: PASSED ✓
  Details: [Verified] Programmatic OK | Checker Model: The actual execution output confirms the creation of all required files (main.py, models.py, requirements.txt) and explicitly states that unit tests were run and passed successfully, which fully satisfies the criteria.

Scenario 2 (Programmatic Failure - Error Caught):
  Result: FAILED ✗ (Caught Programmatically)
  Details: [Programmatic Failed] Output contains error signature: 'traceback (most recent call last)'.

Scenario 3 (Semantic Failure - Checker Model Rejection):
  Result: FAILED ✗ (Caught by Checker Model)
  Details: [Checker-Model Rejected] The actual execution output shows that only an empty main.py file was created and no endpoints, mode

### Task 6: Replanning on Failure

In [ ]:
def replan_on_failure(plan: Plan, failed_step: PlanStep) -> Plan:
    """
    Replans remaining work after a step failure while preserving all completed steps.
    """
    completed = [s for s in plan.steps if s.status == StepStatus.COMPLETED]
    completed_desc = "\n".join(f"- [{s.id}]: {s.description}" for s in completed)

    prompt = f"""The plan for goal '{plan.goal}' encountered a failure.

ALREADY COMPLETED STEPS:
{completed_desc or "None"}

FAILED STEP:
- [{failed_step.id}]: {failed_step.description}
Failure Reason: {failed_step.result_summary}

Task:
Generate replacement steps to achieve the goal from where we are now.
DO NOT add completed plan.
ONLY return new plansteps
"""
    response = google_client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Plan,
            system_instruction="You are a replanning agent. Generate steps to complete the remaining goal, preserving completed progress.",
        ),
    )
    new_plan: Plan = response.parsed

    # Preserve completed steps + append newly generated replacement steps
    plan.steps = completed + new_plan.steps
    plan.version += 1
    return plan


def run_with_replanning(
    plan: Plan,
    executor: Callable[[PlanStep], str],
    max_replans: int = 2,
    verbose: bool = True,
) -> Plan:
    """Executes a plan with automatic replanning on failure, capped at max_replans (default 2)."""
    replans = 0

    while True:
        dag_scheduler(plan, executor=executor, stop_on_failure=True, verbose=verbose)

        failed_step = next((s for s in plan.steps if s.status == StepStatus.FAILED), None)
        if not failed_step:
            if verbose:
                print("\n🎉 Plan finished successfully!")
            return plan

        if replans >= max_replans:
            if verbose:
                print(f"\n⛔ Replan limit reached ({max_replans}/{max_replans}). Halting.")
            return plan

        replans += 1
        if verbose:
            print(
                f"\n🔄 Triggering Replan {replans}/{max_replans} due to failure in [{failed_step.id}] (Version {plan.version} -> {plan.version + 1})...\n"
            )

        plan = replan_on_failure(plan, failed_step)

In [57]:
# Demonstration: Simulate step failure triggering replan (max 2 replans)
test_plan = copy.deepcopy(plan)
reset_plan(test_plan)


# Custom executor that fails once on step_2 to trigger replanning
def flaky_executor(step: PlanStep) -> str:
    if "step_2" in step.id and step.attempts == 1:
        raise RuntimeError("Port 8000 conflict during Docker run. Container failed to start.")
    return f"Done [{step.id}]: {step.description[:40]}"


final_plan = run_with_replanning(test_plan, executor=flaky_executor, max_replans=2, verbose=True)

# Verify completed steps were preserved and version incremented
print(f"\nFinal Plan Version: {final_plan.version}")
print(f"Total Steps in Plan: {len(final_plan.steps)}")
for s in final_plan.steps:
    print(f"  [{s.id}] {s.status.value.upper():<10} - {s.description[:60]}")

=== Starting Sequential DAG Scheduler for: 'Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS' ===

[Iteration 1] Ready steps: ['step_1', 'step_4']
  ▶ Running [step_1]: Develop the FastAPI microservice application code with ... (Root step)
    ✓ COMPLETED: Done [step_1]: Develop the FastAPI microservice applica
  ▶ Running [step_4]: Provision AWS infrastructure (RDS PostgreSQL instance, ... (Root step)
    ✓ COMPLETED: Done [step_4]: Provision AWS infrastructure (RDS Postgr

[Iteration 2] Ready steps: ['step_2']
  ▶ Running [step_2]: Create a multi-stage Dockerfile for containerizing the ... (Depends on: step_1)
    ✗ FAILED: Port 8000 conflict during Docker run. Container failed to start.
  ⛔ Stopping execution due to step failure.


🔄 Triggering Replan 1/2 due to failure in [step_2] (Version 1 -> 2)...

=== Starting Sequential DAG Scheduler for: 'Build and deploy a containerized FastAPI microservice with PostgreSQL and automate

### Task 7: Human Approval & Plan Editing

In [58]:
def human_review_and_approve(
    plan: Plan,
    edit_fn: Callable[[Plan], Plan] | None = None,
    save_path: str = "approved_plan.json",
) -> Plan:
    """
    Human-approval step: allows a reviewer to inspect/edit the plan, validates it,
    and persists the approved plan to disk before execution.
    """
    print(f"=== Reviewing Plan: '{plan.goal}' (v{plan.version}) ===")

    # 1. Apply reviewer edits if provided
    if edit_fn:
        plan = edit_fn(plan)
        print("✏️ Reviewer edits applied.")

    # 2. Validate the approved plan structure and dependencies
    validate_plan(plan)

    # 3. Persist approved version to disk
    Path(save_path).write_text(plan.model_dump_json(indent=2))
    print(f"✅ Approved and persisted to '{save_path}'\n")

    return plan

In [59]:
# Demonstration: Reviewer edits step_1's tool hint, adds an assumption, and approves
def reviewer_edits(p: Plan) -> Plan:
    p.steps[0].tool_name_hint = "Pytest + Ruff + VSCode"
    p.assumptions.append("Reviewer verified AWS IAM permissions are active")
    return p


# Run human approval step
approved_plan = human_review_and_approve(
    copy.deepcopy(plan),
    edit_fn=reviewer_edits,
    save_path="approved_plan.json",
)

# Load persisted version from disk to verify
loaded_plan = Plan.model_validate_json(Path("approved_plan.json").read_text())
print(f"Loaded plan goal: {loaded_plan.goal}")
print(f"Step 1 updated tool: {loaded_plan.steps[0].tool_name_hint}")
print(f"Persisted assumptions: {loaded_plan.assumptions}")

=== Reviewing Plan: 'Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS' (v1) ===
✏️ Reviewer edits applied.
✅ Approved and persisted to 'approved_plan.json'

Loaded plan goal: Build and deploy a containerized FastAPI microservice with PostgreSQL and automated CI/CD pipeline on AWS
Step 1 updated tool: Pytest + Ruff + VSCode
Persisted assumptions: ['AWS account with appropriate permissions is available', 'GitHub repository is configured for storing source code and running Actions', 'Terraform state management backend is accessible', 'Reviewer verified AWS IAM permissions are active']


### Task 8

this will be including in mini-project comparison